# Learning journeys with FastPath, as a predictor

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jose-alvarado-guzman/oulad/blob/main/notebooks/aga_fastpath_journeys.ipynb)

`aga_outcome_prediction.ipynb` embeds students with FastRP, which sees *who* they are connected
to. This one embeds them with **FastPath**, which sees the **shape of the journey**: what they
did, in what order, how long ago, and how hard. FastPath is built for sequences — clickstreams,
customer journeys, event logs — and OULAD's VLE data is a clickstream.

The catch is that the loaded graph has no sequence in it. `REVIEWED_MATERIAL` is a direct
student→material edge carrying a date, with nothing linking one interaction to the next. So this
notebook **builds an event chain first**:

```
(:Student)-[:FIRST_INTERACTION]->(:Interaction)-[:NEXT_INTERACTION]->(:Interaction)-> ...
                                       |
                                 [:OF_MATERIAL]-> (:EducationalMaterial)
```

One `:Interaction` per (student, material, day), chained in date order. FastPath turns each
chain into a vector, and that vector is then judged the only way that answers whether it is
useful: as a **feature for a classifier**, scored on held-out F1 and accuracy against the same
target and the same baseline as the FastRP notebook.

Three variants are trained — journey embedding alone, volume alone, and both — because a single
number in isolation says nothing about whether the embedding earned its keep.

## Before you start

The usual secrets: `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`, `AURA_CLIENT_ID`,
`AURA_CLIENT_SECRET`, `AURA_PROJECT_ID`, with *Notebook access* on. Run
[`oulad_data_load.ipynb`](oulad_data_load.ipynb) first if the graph is not loaded.

> **This notebook writes to your database.** Step 5 creates one `:Interaction` per interaction
> — about 281,000 for the default module, 3.2M for `FFF` — plus two properties on each
> `Student`. **Step 14 deletes all of it.** Nothing else is modified; the OULAD graph is only
> read.

> **A session is billed compute**, separate from AuraDB. Step 14 deletes it; the TTL in step 8
> is only a backstop.

## FastPath is in preview

No Aura-native docs yet. The closest public reference is the Snowflake Graph Analytics
documentation, which covers the same algorithm:
<https://neo4j.com/docs/snowflake-graph-analytics/current/algorithms/fastpath/>

Its Python surface is still moving. Every parameter below was read off
`graphdatascience==2.0a5` directly, and several were renamed from earlier alphas
(`dimension` → `embedding_dimension`, `max_elapsed_time` → `lookback_horizon`,
`num_elapsed_times` → `num_time_anchors`, `time_node_property` →
`event_node_time_property`, `output_time` → `observation_time`, `decay_factor` →
`decay_rate`). Examples written against 2.0a1 will not run as-is.

## 1. Setup

Re-running resets the checkout to `origin/main`, discarding local changes.

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/jose-alvarado-guzman/oulad.git'
REPO_DIR = '/content/oulad'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def run(*command):
    result = subprocess.run(command, text=True, capture_output=True)
    print((result.stdout + result.stderr).strip())
    result.check_returncode()

if IN_COLAB:
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        run('git', '-C', REPO_DIR, 'fetch', '--depth', '1', 'origin', 'main')
        run('git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main')
        run('git', '-C', REPO_DIR, 'clean', '-fd')
    else:
        run('git', 'clone', '--depth', '1', REPO_URL, REPO_DIR)
    run('git', '-C', REPO_DIR, 'log', '-1', '--format=%h %ad %s', '--date=short')
    print()
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r',
         os.path.join(REPO_DIR, 'requirements-aga.txt')], check=True)
    print('Dependencies installed.')
else:
    REPO_DIR = os.getcwd()
    while REPO_DIR != '/' and not os.path.isdir(os.path.join(REPO_DIR, '.git')):
        REPO_DIR = os.path.dirname(REPO_DIR)
    print('Local kernel; assuming requirements-aga.txt is installed.')
    print('Repository root:', REPO_DIR)

## 2. Imports

In [ ]:
import os
import sys
from datetime import timedelta

REPO_DIR = '/content/oulad' if os.path.isdir('/content/oulad') else REPO_DIR
SRC_DIR = os.path.join(REPO_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

for name in [m for m in sys.modules if m == 'oulad' or m.startswith('oulad.')]:
    del sys.modules[name]

import matplotlib.pyplot as plt
import pandas as pd
from neo4j import GraphDatabase
from graphdatascience.session import (
    AlgorithmCategory, AuraAPICredentials, DbmsConnectionInfo, GdsSessions)

import graphdatascience
from oulad.credentials import (
    AGA_SECRETS, ETL_SECRETS, MissingCredentialsError, aura_instance_id, load_credentials)
from oulad.logger import get_logger

print('graphdatascience', graphdatascience.__version__)
print('repository      ', REPO_DIR)

## 3. Credentials and the database connection

In [ ]:
logger = get_logger(REPO_DIR)

try:
    print('resolved from:', load_credentials(logger, required=ETL_SECRETS + AGA_SECRETS))
except MissingCredentialsError as error:
    raise SystemExit(f'\n{error}\n\nAdd the missing secrets in the sidebar, switch on '
                     'Notebook access, then re-run this cell.')

NEO4J_URI = os.environ['NEO4J_URI']
NEO4J_USERNAME = os.environ['NEO4J_USERNAME']
NEO4J_PASSWORD = os.environ['NEO4J_PASSWORD']
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE') or None
AURA_INSTANCE_ID = aura_instance_id(logger)

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()
print('connected to AuraDB, instance', AURA_INSTANCE_ID)

sessions = GdsSessions(api_credentials=AuraAPICredentials(
    os.environ['AURA_CLIENT_ID'], os.environ['AURA_CLIENT_SECRET'],
    os.environ['AURA_PROJECT_ID']))

## 4. Choose the scope and read the time span

One module at a time. The chain is one node per interaction, so the module size *is* the
write size — and every event has to fit inside FastPath's lookback window, which the dates
below determine.

| Module | Students | Events | Avg chain |
| --- | --- | --- | --- |
| `AAA` | 702 | 280,990 | 400 |
| `GGG` | 2,359 | 281,277 | 119 |
| `EEE` | 2,634 | 758,220 | 288 |
| `CCC` | 3,852 | 884,889 | 230 |
| `BBB` | 6,484 | 1,139,085 | 176 |
| `DDD` | 5,407 | 1,866,156 | 345 |
| `FFF` | 6,799 | 3,248,703 | 478 |

`GGG` is the default: enough students for the grouping to mean something, the smallest write,
and short chains. `MAX_STUDENTS` caps the build while you are trying things out — set it to
`None` for the whole module.

OULAD dates are days relative to the module start and can be negative (material viewed before
the course opened), so they are shifted to begin at 0.

In [ ]:
MODULE = 'GGG'
MAX_STUDENTS = None        # e.g. 200 for a quick trial, None for the whole module
CUTOFF_DAY = None          # e.g. 30 to keep only the first 30 days (see step 12)

SPAN_QUERY = '''
MATCH (s:Student)-[r:REVIEWED_MATERIAL]->(m:EducationalMaterial)<-[:HAS_MATERIAL]-(c:Course)
WHERE c.codeModule = $module
RETURN count(DISTINCT s) AS students, count(r) AS events,
       min(r.date) AS minDate, max(r.date) AS maxDate
'''
TYPES_QUERY = '''
MATCH (m:EducationalMaterial)<-[:HAS_MATERIAL]-(c:Course)
WHERE c.codeModule = $module
RETURN DISTINCT m.activityType AS activityType ORDER BY activityType
'''

with driver.session(database=NEO4J_DATABASE) as session:
    span = session.run(SPAN_QUERY, module=MODULE).single()
    activity_types = [r['activityType']
                      for r in session.run(TYPES_QUERY, module=MODULE)]

# GDS node properties must be numeric, so the activity type is encoded as an
# integer. FastPath's event_node_categorical_properties expects that too --
# hence event_node_ignored_category being an int in its signature.
TYPE_IDS = {name: index for index, name in enumerate(activity_types)}

SHIFT = -min(0, span['minDate'])          # move day 0 to the earliest event
last_day = span['maxDate'] if CUTOFF_DAY is None else CUTOFF_DAY
OBSERVATION_TIME = float(last_day + SHIFT + 1)
LOOKBACK_HORIZON = int(OBSERVATION_TIME) + 10   # must exceed the oldest elapsed time
NUM_TIME_ANCHORS = 20

print(f"module {MODULE}: {span['students']:,} students, {span['events']:,} events")
print(f"dates {span['minDate']} to {span['maxDate']}, shifted by +{SHIFT} "
      f'-> 0 to {span["maxDate"] + SHIFT}')
if CUTOFF_DAY is not None:
    print(f'TRUNCATED: only events on or before day {CUTOFF_DAY} '
          f'(shifted {CUTOFF_DAY + SHIFT}) will be chained')
print(f'observation_time {OBSERVATION_TIME:.0f}, lookback_horizon {LOOKBACK_HORIZON}, '
      f'{NUM_TIME_ANCHORS} time anchors '
      f'({LOOKBACK_HORIZON / NUM_TIME_ANCHORS:.1f} days per anchor)')
print(f'\nactivity types encoded: {TYPE_IDS}')

## 5. Build the event chain

**This writes to your database.** One `:Interaction` per (student, material, day) — verified
unique in the source, so no aggregation is needed — chained in date order with ties broken by
material id so the sequence is deterministic.

Batched by student, because one transaction for a whole module is a bad idea. Re-running is
safe: students that already have a chain are skipped, so an interrupted build resumes.

Each `:Interaction` carries `day` (shifted), `clicks`, `activityTypeId`, `seq`, and `module`.
The `module` tag is what makes the scope and the cleanup in step 14 precise.

In [ ]:
BATCH = 100

CANDIDATES_QUERY = '''
MATCH (s:Student)-[r:REVIEWED_MATERIAL]->(:EducationalMaterial)<-[:HAS_MATERIAL]-(c:Course)
WHERE c.codeModule = $module AND NOT (s)-[:FIRST_INTERACTION]->(:Interaction)
  AND ($cutoff IS NULL OR r.date <= $cutoff)
RETURN DISTINCT s.id AS studentId ORDER BY studentId
'''

BUILD_QUERY = '''
UNWIND $studentIds AS studentId
MATCH (s:Student {id: studentId})-[r:REVIEWED_MATERIAL]->(m:EducationalMaterial)
      <-[:HAS_MATERIAL]-(c:Course)
WHERE c.codeModule = $module
  AND ($cutoff IS NULL OR r.date <= $cutoff)
WITH s, m, r ORDER BY r.date, m.id
WITH s, collect({material: m, day: r.date + $shift, clicks: r.sumClick,
                 typeId: $typeIds[m.activityType]}) AS events
UNWIND range(0, size(events) - 1) AS i
WITH s, i, events[i] AS event
// the material has to be bound to its own variable: a node pulled out of a map
// cannot be used directly inside a CREATE pattern
WITH s, i, event.material AS material, event.day AS day,
     event.clicks AS clicks, event.typeId AS typeId
CREATE (ev:Interaction {module: $module, studentId: s.id, seq: i, day: day,
                        clicks: clicks, activityTypeId: typeId,
                        // FastPath reads numeric event features as a vector, so the
                        // click count has to be a list even though it is one number.
                        // Logged: raw totals are heavily skewed.
                        features: [log(toFloat(clicks) + 1.0)]})
CREATE (ev)-[:OF_MATERIAL]->(material)
WITH s, ev ORDER BY ev.seq
WITH s, collect(ev) AS chain
// list elements need binding too, for the same reason as the material above
WITH s, chain, chain[0] AS firstEvent
CREATE (s)-[:FIRST_INTERACTION]->(firstEvent)
WITH chain
UNWIND range(0, size(chain) - 2) AS j
WITH chain[j] AS previous, chain[j + 1] AS following
CREATE (previous)-[:NEXT_INTERACTION]->(following)
RETURN count(*) AS links
'''

with driver.session(database=NEO4J_DATABASE) as session:
    pending = [r['studentId'] for r in session.run(
        CANDIDATES_QUERY, module=MODULE, cutoff=CUTOFF_DAY)]

if MAX_STUDENTS is not None:
    pending = pending[:MAX_STUDENTS]

print(f'{len(pending):,} students still need a chain')
built = 0
for start in range(0, len(pending), BATCH):
    chunk = pending[start:start + BATCH]
    with driver.session(database=NEO4J_DATABASE) as session:
        session.run(BUILD_QUERY, studentIds=chunk, module=MODULE,
                    shift=SHIFT, typeIds=TYPE_IDS, cutoff=CUTOFF_DAY).consume()
    built += len(chunk)
    if built % (BATCH * 5) == 0 or built == len(pending):
        print(f'  {built:,}/{len(pending):,} students', flush=True)
print('chain built' if pending else 'nothing to do, chain already present')

# The classifier needs its target and its baseline feature as numeric node
# properties. They are written onto Student here and removed again in step 14;
# the alternative is computing them in the projection query, which is awkward
# because that query starts from the chain rather than from the student.
LABEL_QUERY = '''
MATCH (s:Student)-[:WAS_REGISTERED]->(:StudentRegistration)-[cc:CONTAINS_COURSE]->(c:Course)
WHERE c.codeModule = $module
WITH DISTINCT s, cc.finalResult AS finalResult
OPTIONAL MATCH (s)-[r:REVIEWED_MATERIAL]->(:EducationalMaterial)<-[:HAS_MATERIAL]-(c2:Course)
WHERE c2.codeModule = $module
WITH s, finalResult, coalesce(sum(r.sumClick), 0) AS clicks
SET s.passed = CASE WHEN finalResult IN $passResults THEN 1 ELSE 0 END,
    s.logClicks = log(toFloat(clicks) + 1.0)
RETURN count(*) AS labelled
'''
PASS_RESULTS = ['Pass', 'Distinction']
with driver.session(database=NEO4J_DATABASE) as session:
    labelled = session.run(LABEL_QUERY, module=MODULE,
                           passResults=PASS_RESULTS).single()['labelled']
print(f'labelled {labelled:,} students with passed and logClicks')


## 6. Check the chain before trusting it

Three things worth confirming, because a broken chain would still embed — just wrongly:

- one `FIRST_INTERACTION` per student with a chain, and no student with two,
- as many `NEXT_INTERACTION` links as events minus students, since each chain of length *n*
  contributes *n-1* links,
- days along a chain never decrease.

In [ ]:
CHECK_QUERY = '''
MATCH (i:Interaction {module: $module})
WITH count(i) AS events
MATCH (:Student)-[f:FIRST_INTERACTION]->(:Interaction {module: $module})
WITH events, count(f) AS firsts
MATCH (:Interaction {module: $module})-[n:NEXT_INTERACTION]->()
RETURN events, firsts, count(n) AS nexts
'''
ORDER_QUERY = '''
MATCH (a:Interaction {module: $module})-[:NEXT_INTERACTION]->(b:Interaction)
WHERE b.day < a.day
RETURN count(*) AS outOfOrder
'''
with driver.session(database=NEO4J_DATABASE) as session:
    counts = session.run(CHECK_QUERY, module=MODULE).single()
    out_of_order = session.run(ORDER_QUERY, module=MODULE).single()['outOfOrder']

expected_nexts = counts['events'] - counts['firsts']
print(f"events {counts['events']:,}, chains {counts['firsts']:,}, "
      f"next links {counts['nexts']:,} (expected {expected_nexts:,})")
print('link count consistent:', counts['nexts'] == expected_nexts)
print('interactions out of date order:', out_of_order)
if counts['nexts'] != expected_nexts or out_of_order:
    raise SystemExit('The chain is not well formed; embedding it would be meaningless. '
                     'Delete it with step 14 and rebuild.')
print('\nchain looks sound')

## 7. A journey, in the raw

Worth seeing what FastPath is being handed before it turns into 128 numbers.

In [ ]:
SAMPLE_QUERY = '''
MATCH (s:Student)-[:FIRST_INTERACTION]->(first:Interaction {module: $module})
WITH s, first ORDER BY s.id LIMIT 1
MATCH path = (first)-[:NEXT_INTERACTION*0..14]->(ev:Interaction)
WITH s, ev ORDER BY ev.seq LIMIT 15
MATCH (ev)-[:OF_MATERIAL]->(m:EducationalMaterial)
RETURN s.id AS studentId, ev.seq AS seq, ev.day AS day,
       m.activityType AS activityType, ev.activityTypeId AS typeId, ev.clicks AS clicks
ORDER BY seq
'''
with driver.session(database=NEO4J_DATABASE) as session:
    sample = pd.DataFrame(session.run(SAMPLE_QUERY, module=MODULE).data())
print(f'first 15 events of one student\'s journey')
print(sample.to_string(index=False))

## 8. Size and open the session

In [ ]:
COUNT_QUERY = '''
MATCH (i:Interaction {module: $module})
WITH count(i) AS events
MATCH (s:Student)-[:FIRST_INTERACTION]->(:Interaction {module: $module})
WITH events, count(DISTINCT s) AS students
MATCH (:Interaction {module: $module})-[n:NEXT_INTERACTION]->()
RETURN events, students, events + count(n) AS relationships
'''
with driver.session(database=NEO4J_DATABASE) as session:
    sized = session.run(COUNT_QUERY, module=MODULE).single()

node_count = sized['events'] + sized['students']
relationship_count = sized['relationships']
print(f"{sized['students']:,} students + {sized['events']:,} interactions "
      f'= {node_count:,} nodes, {relationship_count:,} relationships')

memory = sessions.estimate(
    node_count=node_count,
    relationship_count=relationship_count,
    algorithm_categories=[AlgorithmCategory.NODE_EMBEDDING,
                          AlgorithmCategory.SIMILARITY,
                          AlgorithmCategory.COMMUNITY_DETECTION],
    node_label_count=2,          # Student, Interaction
    node_property_count=4,       # id, day, clicks, activityTypeId
)
print('estimated memory:', memory)

SESSION_NAME = f"oulad-fastpath-{os.environ['AURA_CLIENT_ID'][:8]}"
gds = sessions.get_or_create(
    session_name=SESSION_NAME,
    memory=memory,
    db_connection=DbmsConnectionInfo(
        aura_instance_id=AURA_INSTANCE_ID,
        username=NEO4J_USERNAME, password=NEO4J_PASSWORD, database=NEO4J_DATABASE),
    ttl=timedelta(hours=2),
)
print('session ready:', SESSION_NAME)

## 9. Project the chain

Both chain relationship types in one query. The source of a row is a `Student` for
`FIRST_INTERACTION` and an `Interaction` for `NEXT_INTERACTION`, so the property map asks for
the union of both — a node simply has no value for the keys that do not apply to it.

Everything FastPath reads has to be numeric, which is why `activityTypeId` is projected rather
than `activityType`.

In [ ]:
GRAPH_NAME = 'oulad-journeys'

PROJECTION_QUERY = '''
MATCH (src)-[r:FIRST_INTERACTION|NEXT_INTERACTION]->(tgt:Interaction)
WHERE tgt.module = $module
RETURN gds.graph.project.remote(src, tgt, {
    sourceNodeLabels: labels(src),
    targetNodeLabels: labels(tgt),
    sourceNodeProperties: src { .id, .day, .clicks, .activityTypeId, .features, .passed, .logClicks },
    targetNodeProperties: tgt { .day, .clicks, .activityTypeId, .features },
    relationshipType: type(r)
})
'''

gds.graph.project.cypher(
    graph_name=GRAPH_NAME, query=PROJECTION_QUERY,
    query_parameters={'module': MODULE}, overwrite=True)
G = gds.graph.get(GRAPH_NAME)
print(f'projected {G.node_count():,} nodes and {G.relationship_count():,} relationships')
print('node properties        :', G.node_properties())
print('relationship properties:', G.relationship_properties())

for label, needed in [('Interaction', {'day', 'activityTypeId', 'features'}),
                      ('Student', {'passed', 'logClicks'})]:
    missing = needed - set(G.node_properties().get(label, []))
    if missing:
        raise SystemExit(f'{label} is missing {missing} in the projection; FastPath would '
                         'either fail or read defaults. Check the projection query.')
print('\nthe properties FastPath needs are present')

## 10. FastPath embeddings

Each student's chain becomes one vector. The parameters worth understanding:

- **`observation_time`** is the vantage point. Elapsed time is measured back from here, and
  events at or after it are excluded — so it is set past the last event.
- **`lookback_horizon`** is how far back to look; it must exceed the oldest elapsed time or the
  earliest events fall outside the window and are dropped.
- **`num_time_anchors`** buckets that window. Twenty anchors over ~296 days is a fortnight
  each: enough to tell "worked steadily" from "crammed at the end".
- **`event_node_categorical_properties`** is what the events are *made of* — the kind of
  material touched.
- **`event_node_feature_vector_property`** is *how much* each event was worth. This carries
  the logged click count.
- **`random_seed`** is fixed so the embedding is reproducible.

> The feature vector was missing from the first version of this notebook, and its absence
> mattered: `sumClick` was copied onto the event nodes and projected into the session, then
> never passed to the algorithm. A page opened once and a page hammered fifty times were the
> same event. Any earlier result should be read as a different configuration, not as a verdict
> on FastPath.

In [ ]:
EMBEDDING_PROPERTY = 'journeyEmbedding'

embedding = gds.fast_path.mutate(
    G,
    base_node_label='Student',
    event_node_label='Interaction',
    mutate_property=EMBEDDING_PROPERTY,
    embedding_dimension=128,
    lookback_horizon=LOOKBACK_HORIZON,
    num_time_anchors=NUM_TIME_ANCHORS,
    event_node_categorical_properties=['activityTypeId'],
    event_node_feature_vector_property='features',   # click intensity
    event_node_time_property='day',
    first_relationship_type='FIRST_INTERACTION',
    next_relationship_type='NEXT_INTERACTION',
    observation_time=OBSERVATION_TIME,
    smoothing_window=2,
    smoothing_rate=10.0 / LOOKBACK_HORIZON,
    random_seed=42,
)
print(embedding)

## 11. Does the journey embedding predict the outcome?

The question a clustering score cannot answer. Louvain modularity says how cleanly the
similarity graph splits; it says nothing about whether the split is *useful*. So the embedding
goes into a **node classification pipeline** instead, and is judged on held-out F1 and accuracy
— the same measures, on the same module, as `aga_outcome_prediction.ipynb` uses for FastRP.

Three variants, because only the comparison is informative:

| variant | what it answers |
| --- | --- |
| journey embedding only | does the *sequence* carry predictive signal at all? |
| volume only | what does one logged aggregate already achieve? |
| journey + volume | does sequence add anything on top of volume? |

The embedding is already on the Student nodes from step 10, so no node-property step is needed
inside the pipeline — `select_features` names it directly.

In [ ]:
def train_variant(features, suffix):
    """Train the pipeline over `features` and return its held-out metrics."""
    pipeline_name = f'fastpath-pipeline-{suffix}'
    model_name = f'fastpath-model-{suffix}'
    for drop in (lambda: gds.model.get(model_name).drop(),
                 lambda: gds.pipeline.node_classification.get(pipeline_name).drop()):
        try:
            drop()
        except Exception:
            pass

    pipe, _ = gds.pipeline.node_classification.create(pipeline_name)
    pipe.select_features(features)
    pipe.configure_split(test_fraction=0.3, validation_folds=4)
    pipe.add_logistic_regression(penalty=(0.001, 1.0), max_epochs=300)
    pipe.add_random_forest(max_depth=(4, 16), number_of_decision_trees=200)

    model, _ = pipe.train(
        G, model_name=model_name, metrics=['F1_MACRO', 'ACCURACY'],
        target_property='passed', target_node_labels=['Student'], random_seed=42)
    scores = model.metrics() or {}
    method = (model.best_parameters() or {}).get('methodName', '?')
    return model, {m: v.get('test') for m, v in scores.items()
                   if isinstance(v, dict)}, method

VARIANTS = {
    'journey embedding only': ([EMBEDDING_PROPERTY], 'journey'),
    'volume only':            (['logClicks'], 'volume'),
    'journey + volume':       ([EMBEDDING_PROPERTY, 'logClicks'], 'both'),
}

models, results, winners = {}, {}, {}
for label, (features, suffix) in VARIANTS.items():
    print(f'\ntraining: {label}  features={features}', flush=True)
    model, scores, method = train_variant(features, suffix)
    models[label], results[label], winners[label] = model, scores, method
    print(f'  {scores}  (winner: {method})', flush=True)

## 12. The comparison

The row that matters is **journey + volume against volume only**. If the sequence embedding
carries information a single aggregate does not, that gap is where it shows up.

A threshold baseline is included too — predict pass above the median click count, no model at
all — scored over the same students the models see, since a student with no interactions has no
chain and so appears in neither.

> ### Read this before believing the number
>
> With `CUTOFF_DAY = None` the embedding sees each student's **whole** journey, right up to the
> end of the presentation. That includes *when the activity stopped* — and for a student who
> withdrew, when the activity stopped is very nearly the label itself. A strong score here is a
> real retrospective classification, but it is not evidence that anyone could have been warned
> in time.
>
> The honest early-warning test is `CUTOFF_DAY = 30`: rebuild the chain from the first 30 days
> only and score again. Whatever survives that truncation is signal you could actually have
> acted on. The gap between the two runs is the part that was hindsight.

In [ ]:
BASELINE_QUERY = '''
MATCH (s:Student)-[:WAS_REGISTERED]->(:StudentRegistration)-[cc:CONTAINS_COURSE]->(c:Course)
WHERE c.codeModule = $module
WITH DISTINCT s, cc.finalResult AS finalResult
// An inner match on purpose: a student with no interactions has no chain, so is
// not in the projection and not in any trained model. Scoring the baseline over
// a larger population than the models see would not be a comparison.
MATCH (s)-[r:REVIEWED_MATERIAL]->(:EducationalMaterial)<-[:HAS_MATERIAL]-(c2:Course)
WHERE c2.codeModule = $module
RETURN s.id AS studentId,
       CASE WHEN finalResult IN $passResults THEN 1 ELSE 0 END AS passed,
       sum(r.sumClick) AS clicks
'''
with driver.session(database=NEO4J_DATABASE) as session:
    base = (pd.DataFrame(session.run(
        BASELINE_QUERY, module=MODULE, passResults=PASS_RESULTS).data())
            .drop_duplicates(subset='studentId'))

def macro_f1(truth, predicted):
    scores = []
    for label in (0, 1):
        tp = ((predicted == label) & (truth == label)).sum()
        fp = ((predicted == label) & (truth != label)).sum()
        fn = ((predicted != label) & (truth == label)).sum()
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        scores.append(2 * precision * recall / (precision + recall)
                      if precision + recall else 0.0)
    return sum(scores) / len(scores)

predicted = (base['clicks'] >= base['clicks'].median()).astype(int)
BASELINE = {'F1_MACRO': macro_f1(base['passed'], predicted),
            'ACCURACY': (predicted == base['passed']).mean()}
majority = max(base['passed'].mean(), 1 - base['passed'].mean())

rows = [{'method': label, **scores, 'winner': winners[label]}
        for label, scores in results.items()]
rows.append({'method': 'clicks >= median (no model)', **BASELINE, 'winner': '-'})
rows.append({'method': 'always predict majority', 'F1_MACRO': None,
             'ACCURACY': majority, 'winner': '-'})
print(pd.DataFrame(rows).set_index('method').to_string())

both = results.get('journey + volume', {}).get('ACCURACY')
volume = results.get('volume only', {}).get('ACCURACY')
journey = results.get('journey embedding only', {}).get('ACCURACY')
if both and volume:
    gain = (both - volume) * 100
    print(f'\njourney embedding adds {gain:+.2f} accuracy points on top of volume')
    print('  within noise on this test split' if abs(gain) < 1.5
          else '  a real difference on this test split')
if journey:
    print(f'sequence alone reaches {journey:.4f} accuracy '
          f'against {majority:.4f} for always guessing the majority class')

## 13. Where the best model is wrong

A confusion matrix over every labelled student. For an early-warning use the interesting cell
is bottom-left: students who failed or withdrew and were *not* flagged.

In [ ]:
best_label = max(results, key=lambda k: results[k].get('ACCURACY') or 0)
print(f'best variant: {best_label}')
best = models[best_label]

predictions = best.predict_stream(G, target_node_labels=['Student'])
truth = gds.graph.node_properties.stream(G, 'passed', node_labels=['Student'])
truth_column = [c for c in truth.columns if c != 'nodeId'][-1]
predicted_column = [c for c in predictions.columns
                    if c != 'nodeId' and 'probab' not in c.lower()][-1]

merged = (predictions.rename(columns={predicted_column: 'predicted'})[['nodeId', 'predicted']]
          .merge(truth.rename(columns={truth_column: 'actual'})[['nodeId', 'actual']],
                 on='nodeId'))
print()
print(pd.crosstab(merged['actual'], merged['predicted'],
                  rownames=['actual'], colnames=['predicted']).to_string())

caught = ((merged['actual'] == 0) & (merged['predicted'] == 0)).sum()
at_risk = (merged['actual'] == 0).sum()
print(f'\nat-risk students flagged: {caught:,} of {at_risk:,} '
      f'({caught / at_risk * 100:.1f}%)')
print(f"overall agreement: {(merged['predicted'] == merged['actual']).mean():.4f}")

## 14. Clean up

Three things to release, and all of them matter: the trained models and pipelines, the session
(billed compute), and the event chain plus the two `Student` properties, which are the only
marks this notebook leaves on your database.

`DELETE_CHAIN` defaults to `True`. A property on existing nodes is cheap to leave behind; a few
hundred thousand extra nodes is not. Set it to `False` to keep the chain for another run — step
5 skips students that already have one.

In [ ]:
DELETE_CHAIN = True

for label, (_, suffix) in VARIANTS.items():
    for what, drop in [
        ('model', lambda s=suffix: gds.model.get(f'fastpath-model-{s}').drop()),
        ('pipeline', lambda s=suffix: gds.pipeline.node_classification.get(
            f'fastpath-pipeline-{s}').drop()),
    ]:
        try:
            drop()
        except Exception as error:
            print(f'{what} {suffix}: {str(error)[:70]}')
print('models and pipelines released')

try:
    G.drop(); print('projection dropped')
except Exception as error:
    print('projection:', error)
try:
    gds.delete(); print('session deleted')
except Exception as error:
    print('session:', error)

if DELETE_CHAIN:
    # Batched: DETACH DELETE over a few hundred thousand nodes in one
    # transaction is how you run a session out of memory.
    DELETE_QUERY = '''
    MATCH (i:Interaction {module: $module})
    WITH i LIMIT $batch
    DETACH DELETE i
    RETURN count(*) AS deleted
    '''
    removed = 0
    while True:
        with driver.session(database=NEO4J_DATABASE) as session:
            deleted = session.run(DELETE_QUERY, module=MODULE, batch=10000).single()['deleted']
        removed += deleted
        if deleted:
            print(f'  deleted {removed:,}', flush=True)
        if deleted == 0:
            break
    print(f'removed {removed:,} interaction nodes')
    with driver.session(database=NEO4J_DATABASE) as session:
        session.run('MATCH (s:Student) WHERE s.passed IS NOT NULL '
                    'REMOVE s.passed, s.logClicks').consume()
    print('removed the passed and logClicks properties from Student')
else:
    print('DELETE_CHAIN is False; the event chain is still in the database')

with driver.session(database=NEO4J_DATABASE) as session:
    left = session.run('MATCH (i:Interaction) RETURN count(i) AS n').single()['n']
    totals = session.run(
        'MATCH (n) WITH count(n) AS nodes '
        'MATCH ()-[r]->() RETURN nodes, count(r) AS relationships').single()
driver.close()

print(f'\ninteraction nodes remaining: {left:,}')
print(f"graph totals: {totals['nodes']:,} nodes, {totals['relationships']:,} relationships")
print('(66,920 nodes and 8,818,076 relationships is the untouched OULAD graph)')

## What the runs showed

Full module GGG, 2,359 students with a chain, FastPath embeddings fed to a node classification
pipeline and scored on a held-out 30% split:

| features | F1 macro | accuracy |
| --- | --- | --- |
| always predict majority | — | 0.6393 |
| clicks ≥ median, no model | 0.7635 | 0.7681 |
| volume only | 0.8436 | 0.8658 |
| journey embedding only | 0.9176 | 0.9280 |
| **journey + volume** | **0.9214** | **0.9308** |

The sequence embedding is worth **+6.5 accuracy points** over volume alone, and volume is worth
only +0.3 on top of the embedding. The confusion matrix flags 707 of 845 at-risk students, 83.7%.

Compare `aga_outcome_prediction.ipynb`, where a FastRP *topology* embedding on the same module
and target was worth +0.4 points over volume, and collapsed to a constant classifier on its own.
Sequence carries information that neither topology nor a single aggregate does.

## The caveat that decides what this is worth

With `CUTOFF_DAY = None` the embedding sees the whole journey, including when activity stopped —
which for a withdrawal is close to the label itself. So 0.9308 is a sound retrospective
classification and *not* evidence of early warning. Set `CUTOFF_DAY = 30`, rebuild, and rerun:
what survives is what could actually have been acted on, and the difference between the two runs
is the part that was hindsight.

## Why modularity was the wrong measure

Two earlier runs were judged on Louvain modularity — 0.5470 without click intensity, 0.5122 with
it — and both readings suggested the approach was not working. Modularity scores how cleanly a
similarity graph partitions, which is not the same question as whether the embedding predicts
anything. Measured as a classifier feature, the same embedding is the strongest predictor in this
repository. Pick the measure that matches the question.

## Where to look next

`MODULE` in step 4. GGG has the shortest chains of any OULAD module (average 119 events); `FFF`
averages 478 across 3.2M interactions, so journeys have far more room to differ in shape — at the
cost of a much larger build.